In [ ]:
MK                 = None
size               = None
ud                 = None
od                 = None
times_we_tried_max = None
file_path_Lvl      = None
file_path_MK       = None
output_count       = None
save1perc          = None
override_uod       = None
ud_lvl             = None
od_lvl             = None

In [28]:
%run ./7___Finder___Functions.ipynb

In [29]:
progress_file = "./progress/"+str(output_count)+".log"

logging.basicConfig(filename=progress_file,
                    level=logging.INFO,
                    format="%(message)s")

---
---
---

## Load dictionaries

In [30]:
if not override_uod:
    with open(file_path_Lvl+"ud_lvl___"+str(ud)+".pk", 'rb') as f: ud_lvl = pkl.load(f)
    with open(file_path_Lvl+"od_lvl___"+str(od)+".pk", 'rb') as f: od_lvl = pkl.load(f)

with open(file_path_Lvl+"dict_origins_isolated.pk", 'rb') as f: dict_origins_isolated = pkl.load(f)
with open(file_path_Lvl+"dict_origins_pairs.pk",    'rb') as f: dict_origins_pairs    = pkl.load(f)
lvl = len(dict_origins_isolated)

---

### Voids to be merged

In [31]:
# Filled grid - with the indices of each void
fg = np.zeros((size, size, size), dtype=int) - 1

In [32]:
origins_isolated_ud = []; origins_pairs_ud = []
origins_isolated_ud_index = []; origins_pairs_ud_index = []

void_index_od = 0
found_merge = False

void_index = -1
for lvl_i in range(lvl):

    if len(dict_origins_isolated[lvl_i]) != 0:
        for isolated_i in dict_origins_isolated[lvl_i]:
            void_index += 1
            if lvl_i < ud_lvl:
                origins_isolated_ud.append(      isolated_i)
                origins_isolated_ud_index.append(void_index)
            fg[tuple(isolated_i)] = void_index
            if (not found_merge) and (od_lvl <= lvl_i):
                void_index_od = void_index
                found_merge = True

    if MK == "MK2":
        if len(dict_origins_pairs[lvl_i]) != 0:
            for pairs_i in dict_origins_pairs[lvl_i]:
                void_index += 1
                if lvl_i < ud_lvl:
                    origins_pairs_ud.append(      pairs_i[0])
                    origins_pairs_ud_index.append(void_index)
                for pairs_ij in pairs_i:
                    fg[tuple(pairs_ij)] = void_index
                if (not found_merge) and (od_lvl <= lvl_i):
                    void_index_od = void_index
                    found_merge = True

no_voids = void_index+1

In [33]:
origins_isolated_ud       = np.array(origins_isolated_ud)
origins_isolated_ud_index = np.array(origins_isolated_ud_index)
origins_pairs_ud          = np.array(origins_pairs_ud)
origins_pairs_ud_index    = np.array(origins_pairs_ud_index)

In [34]:
with open(file_path_Lvl+"fg___0_"+MK+".pk", 'wb') as f: pkl.dump(fg, f)

---

### ud - patches

If we are not interested into seeing the buildup process, we can fill-in all the cells in the ud regime and simply make patches from them.

This is a major time-saver and leads to identical results (since all these cells form voids that merge into-one-another, irrespective of whether they are also in od regime or not).

In [35]:
if not save1perc:

    # Get the leveled grid.
    with open(file_path_Lvl+"grid_lvld.pk", 'rb') as f: grid_lvld = pkl.load(f)

    if MK == "MK1":
        origins_pairs_ud = np.array([[0,0,0]])
        origins_pairs_ud_index = np.array([0])
    
    ud_patches(grid_lvld, ud_lvl, origins_isolated_ud, origins_isolated_ud_index, origins_pairs_ud, origins_pairs_ud_index, size, MK)

    # fill-in the other void origins (non-ud)
    mask = (fg != -1) & (grid_lvld == -1)
    grid_lvld[mask] = fg[mask]
    fg = grid_lvld

    del mask; gc.collect()

---
---
---
---
---
---

### Run the loop

In [36]:
ratio_perc = 100
lvl_perc   = lvl // ratio_perc

# We found this to be a good limit given our RAM.
max_lvls_per_list = round(lvl / 100 * (512/size)**3)
no_lists = lvl // max_lvls_per_list + (1 if (lvl % max_lvls_per_list)!=0 else 0)

---

In [11]:
walls_to_be_removed = []
remaining_ods = True


for i in tqdm(range(no_lists), desc="Processing"):
    
    with open(file_path_Lvl+"dict_cells_isolated/"+str(i)+".pk", 'rb') as f: dict_cells_isolated = pkl.load(f)
    with open(file_path_Lvl+"dict_cells_pairs/"+str(i)   +".pk", 'rb') as f: dict_cells_pairs    = pkl.load(f)
    
    start_lvl = max_lvls_per_list*i
    end_lvl   = np.min([max_lvls_per_list*(i+1), lvl])

    for current_lvl in range(start_lvl, end_lvl):

        # In case we created the ud pathches grid before and if so, to make sure we're starting from the next level.
        if save1perc or (current_lvl >= ud_lvl):
            
            if save1perc and (current_lvl != 0) and (current_lvl % lvl_perc == 0):
                with open(file_path_MK+"fg___"+str(current_lvl // lvl_perc)+".pk", 'wb') as f: pkl.dump(fg, f)
            
            ud = False; od = False
            if current_lvl < ud_lvl: ud = True
    
            # If we are in od regime and have od's left...
            if (current_lvl >= od_lvl) and remaining_ods:
                # We check agian if we have od's left.
                if (np.max(fg) >= void_index_od): od = True
                else: remaining_ods = False
                    
    
            # List of all neighbor values of each od void... where we limit ourselves to 1 since for more than that we use them as pairs.
            #   -1 - no neighbor / already solved this one
            # >= 0 - the index of the one void
            #   -2 - multiple neighbors
            n_od_voids = no_voids - void_index_od
            od_connections_all_val = np.full(n_od_voids, -1, dtype=np.int64)
    
    
    
            
            # First, we remove all isolated origins from the level isolated dict.
            for i0 in dict_origins_isolated[current_lvl]: dict_cells_isolated[current_lvl].remove(i0)
            
            # If we're using MK2, we must remove the pair origins from the pair dictionary.
            if MK == "MK2":
                for pairs_i in dict_origins_pairs[current_lvl]:
                    dict_cells_pairs[current_lvl] = [coords for coords in dict_cells_pairs[current_lvl] 
                                                     if set(map(tuple, coords)) != set(map(tuple, pairs_i))]
            
            
            
            # At this point, we can run the isolated cells.
            isolated_i = np.array(dict_cells_isolated[current_lvl])
            if len(isolated_i) != 0:
                walls_to_be_removed_loop = loop_isolated(fg, isolated_i, od_connections_all_val, True, ud, od, void_index_od, size)
                for wtbrl in walls_to_be_removed_loop: walls_to_be_removed.append(wtbrl)
    
            # If we're using MK1 or MK2 in ud regime, we take those pairs and run them as isolated cells.
            # We run the loop_isolated pair by pair... faster this way.
            if MK == "MK1" or ud:
                for pairs_i in dict_cells_pairs[current_lvl]:
                    walls_to_be_removed_loop = loop_isolated(fg, np.array(pairs_i), od_connections_all_val, False, ud, od, void_index_od, size)
                    for wtbrl in walls_to_be_removed_loop: walls_to_be_removed.append(wtbrl)
                    
                
    
    
    
            # RUN THE FINDER
            
            # Before we start, how do we deal with potholes?
            # A pothole is an od void that essentially we need to consider as a pothole that we need to fill up to the 
            #     current level and then let non-od voids fill it in as they touch it.
            # - If one void touches it in a level, then the whole od void becomes part of it.
            # - If multiple voids touch it in a level, then we need to consider that void as a pothole filled up to that
            #     level and that now we run loop_pairs on it.
    
            # BUT WE CAN'T KNOW A PRIORI IF AN OD VOID ONLY TOUCHES ONE VOID (by the end of the level).
            # This statement is the foundation for the following procedure in managing od's.
    
            # To achieve, this, we must fist run the whole level (isolated and pairs cells) and see which od voids are connected.
            # Of course, if two od voids neighbor each-other, they must first be merged (and all the connections of the later)
            #     now become part of the former's.
    
            # Sure, ideally, it would be best if we would find a connection within a pair and then start going into the od with the
            #     regular void. Otherwise, we give the other voids that have a longer path through this pair a head-start: it's like
            #     they have an fair race through the path, but it doesn't matter which one reached the od first, they all start
            #     chewing through it at the same time.
            # This is true... but the effect is generally small and the times this occurs are rare at most... and implementing an
            #     ideal code would require to do the isolated ones like above, then run all pairs for one peel, stop, check for 
            #     connections, then merge all isolated and/or pair cells to the (possibly multiple) od('s), then repeat this 
            #     stop-and-go for every peel (which is now for all pairs at once). All this would add so much processing time for
            #     little to no actual benefit.
            # Plus, this is all a matter of interpretation: what if, instead, we considered that we fill-in the od's based on the
            #     steepness and distance from a regular's connection to it and its center... like a race to the bottom: it is 
            #     subjective (or at least a different type of paper needs to study this beforehand, but it'd most likely be a waste
            #     of everyone's time).
    
         
    
            # The pair cells are a bit more tricky....
            # First of all, if we are before the od's or we ran out of them, we can just run the regular loop_pair.
            if (not ud) and (MK == "MK2"):
                pairs = dict_cells_pairs[current_lvl]
                if len(pairs) != 0:
                    for pairs_i in pairs:
                        pairs_i = np.array(pairs_i)
                        walls_to_be_removed_loop = []
        
                        # For starters, it would be a great time-saver to find-out if a pair only touches a single void... then we can
                        #     just attribute the whole pair to it.
                        exterior_outline_vals = check_neighbors_cube(fg, pairs_i, size, void_index_od)
                        
                        # If it touches no voids, it is isolated... becomes a wall.
                        if   len(exterior_outline_vals) == 0:
                            for [i,j,k] in pairs_i:
                                fg[i][j][k] = -3
                                walls_to_be_removed.append([i,j,k])
        
                        # If it only touches one single (regular or od) void, it is part of that void.
                        elif len(exterior_outline_vals) == 1:
                            eov0 = exterior_outline_vals[0]
                            for [i,j,k] in pairs_i: fg[i][j][k] = eov0
        
                            
    
                        
                        # If it touches multiple voids, then it might be time for loop_pairs.
                        else:
                            
                            if not od:
                                # If we are not in od regime, we just run the loop_pair as usual.
                                # It's faster than looking at how many neighbours there are and risking to repeat it all again inside the loop.
                                # Also, the initially_ignore_od parameter is set to False because if we are not in od regime, we haven't reached the od voids' origins,
                                #     let alone make connections with them... so we set it to false so we don't later hope ods can save the day.
                                walls_to_be_removed_loop = loop_pairs(fg, pairs_i, od_connections_all_val, size, False, void_index_od, times_we_tried_max=times_we_tried_max)
                                
                            
                            else:
                                # If we may have od's, then it gets tricky...
                                # We perform the same counting of neighbors as above, but now we separate them into regualars and od's.
                                # There are many such cases and below we list them all.
            
                                # Get all pair's neighbors and their values.
                                exterior_outline_vals_regular = [_ for _ in exterior_outline_vals if _ <  void_index_od]
                                exterior_outline_vals_od      = [_ for _ in exterior_outline_vals if _ >= void_index_od]
    
                                
                                # If it touches no od voids, we run loop_pairs.
                                if   len(exterior_outline_vals_od) == 0:
                                    # Again, we set the initially_ignore_od parameter to False, as to not hope ods can save us if the permutaion fails.
                                    walls_to_be_removed_loop = loop_pairs(fg, pairs_i, od_connections_all_val, size, False, void_index_od, times_we_tried_max=times_we_tried_max)
                
                                # If it touches no regular voids, we simply add the pair to the first od (not that it matters) and merge the rest (since
                                #     they all connect through the pair).
                                elif len(exterior_outline_vals_regular) == 0:
                                    # Merge the pair into the first od.
                                    eov0 = exterior_outline_vals_od[0]
                                    for [i,j,k] in pairs_i:
                                        fg[i][j][k] = eov0
                                    
                                    # We start from the first od's current connection value.
                                    # This can be:
                                    #   -1  -> it does not yet touch any regular
                                    #  >=0  -> it touches one specific regular
                                    #   -2  -> it already touches multiple regulars
                                    odcav0 = od_connections_all_val[eov0-void_index_od]
                                    
                                    # For the other od's it touches...
                                    for eov in exterior_outline_vals_od[1:]:
                                    
                                        # get that od's connection value...
                                        odcav = od_connections_all_val[eov-void_index_od]
                                    
                                        # and combine it with the first od's connection value.
                                        # This preserves the case where both od's touch the same regular, and only sets -2
                                        #     if they touch different regulars or if one of them already had multiple connections.
                                        odcav0 = combine_od_connection(odcav0, odcav)
                                    
                                        # Then merge this od into the first od...
                                        replacement_single(fg, eov, eov0, size)
                                    
                                        # and set it in the connections list as solved, since it no longer exists as its own od void.
                                        od_connections_all_val[eov-void_index_od] = -1
                                    
                                    # Finally, store the combined connection value back into the surviving od.
                                    od_connections_all_val[eov0-void_index_od] = odcav0
    
                                
                                # But if it touches a mix of a regular and (possibly multiple) od voids... we need to be careful.
                                # 
                                # We want to preserve the pothole meaning: it is a valley which we fill in the very last moment before going to the next level.
                                # The reason we fill it at the end of the level is two-fold:
                                #     1. We must first find all its regular connections (and od ones to merge them into a bigger pothole).
                                #     2. We treat this level's connections from regulars to it as the last bridge they must pass: it is not the pothole that can
                                #            still extend towards the regulars, but the regulars that first extend to the pothole: i.e. the pothole should end 
                                #            below this level, not be part of it.
                                # And so, if we can fill this pair with regulars-only, we very much prefer that.
                                else:
                                    # If it touches just one regular (and one or more od's), then the pair belongs to the regular.
                                    if len(exterior_outline_vals_regular) == 1:
                                        eov0 = exterior_outline_vals_regular[0]
                                
                                        for [i,j,k] in pairs_i: fg[i][j][k] = eov0
                                
                                    # If it touches multiple regulars, first try to partition it using regular voids only.
                                    else:
                                        walls_to_be_removed_loop = loop_pairs(fg, pairs_i, od_connections_all_val, size, True, void_index_od, times_we_tried_max=times_we_tried_max )
                                    
                                    register_od_regular_touch_from_cells(fg, pairs_i, od_connections_all_val, size, void_index_od )
                            
                            for wtbrl in walls_to_be_removed_loop: walls_to_be_removed.append(wtbrl)

     
            # POTHOLES
            # We're done managing this level'ls pairs/isolated cells.
            # All that is left is, through the connections they may now have made, to take care of the od voids we might have encountered.
            if (not ud) and od:
                # We go od void by od void...
                for i0 in range(n_od_voids):
                    index0 = void_index_od+i0
    
                    # If one touches a (single) (non-od) void, it becomes part of it...
                    if   od_connections_all_val[i0] >=  0:
                        replacement_single(fg, index0, od_connections_all_val[i0], size)
                        # and set them as solved.
                        od_connections_all_val[i0] = -1
    
                    # If it touches multiple voids, we run it as a pair though the loop_pairs function....
                    elif od_connections_all_val[i0] == -2:
                        # Get all cells belonging to this od pothole.
                        pairs_i = np.array(argwhere_single(fg, index0, size))
                    
                        # Since we now want to refill this od as a flat current-level pair/pothole, we must first remove its own value from the grid.
                        # Otherwise loop_pairs() sees the pothole itself as a neighbouring void and may simply glue cells back to the same od index, leaving it alive.
                        for [i,j,k] in pairs_i: fg[i][j][k] = -1
                    
                        if MK == "MK2": walls_to_be_removed_loop = loop_pairs(fg, pairs_i, od_connections_all_val, size, False, void_index_od, times_we_tried_max=1 )
                    
                        else:           walls_to_be_removed_loop = loop_isolated_leftovers(fg, pairs_i, size, void_index_od )

                        # Those are new walls_to_be_removed from the od's.
                        for wtbrl in walls_to_be_removed_loop: walls_to_be_removed.append(wtbrl)
                        # and set them as solved.
                        od_connections_all_val[i0] = -1
    
    
            # Finally, if we are using MK2, we have set the isolated cells as -3's. This way, we can quickly find them in the grid
            #     so we can try to trim them out.
            # Of course, you cannot do that to one that was just added, since there has not yet been any new cell next to it that would
            #     become a void, thus allowing it to do the same.
            if MK == "MK2" and len(walls_to_be_removed) != 0:
                fg, walls_to_be_removed = thin_walls(fg, np.array(walls_to_be_removed), size, void_index_od)
            
            # For the Monitor_Progress_Finder.ipynb notebook.
            logging.info(f"{int((current_lvl+1) // lvl_perc)}")

Processing: 100%|███████████████████████████████| 13/13 [02:30<00:00, 11.58s/it]


---

### Thin the walls for MK1, Replace Uncertain Walls & Save

In [12]:
# We did this for MK2 at the end of every level...
if MK == "MK1" and len(walls_to_be_removed) != 0: fg, walls_to_be_removed = thin_walls(fg, np.array(walls_to_be_removed), size, void_index_od)

In [13]:
# However, there may be some cells like from a pair or an od that had a narrow area to tie themselves to regulars, but that got transformed
#     into a wall (-2), thus isolating the rest.
# It's like making a run out of the cave with your boyz and the first one sees Medusa outside so he becomes stone and closes the cave for
#     all of you behind him.
replacement_single(fg, -3, -2, size)

# Also, it may be that some leftovers that became -3 walls did afterwards become proper walls: they ended-up separating two voids, even if
#     incorrectly so, as the voids' cells would be a higher level than it... but this would require a massibe backtracking at the time that
#     occurs and virtually never happens and has little to no benefits.

# Really after this we should use thin_walls() where we now look for all the walls that could be removed because they do not separate two
#     different voids... but this situation is so rare and just not worth the massive computational time.
# It is rare because it isn't the previously -3 now -2 wall cells that can be thinned, but the previously (and still) -2 ones that
#     would have formed the walls to the cave to stick to the metaphor. And after we erode them we may erode inside the cave (the ex -3's).

In [14]:
with open(file_path_MK+"fg___100.pk", 'wb') as f: pkl.dump(fg, f)

---
---
---